## 절대 모멘텀
- 전년도(3개월, 6개월 12개월)의 수정주가와 전월의 수정주가를 이용하여 구매의 타이밍을 잡는 투자 전략
- 구매 신호 -> (전월의 수정 주가 / 전년도의 수정 주가) - 1 값이 0보다 크고 무한대가 아닌 경우

1. 파생변수 STD-YM 생성 : index에서 년-월을 추출하여 대입
2. STD-YM별 마지막날의 데이터들을 모아서 month_last_df 데이터 프레임을 생성
3. 전월의 수정주가 파생변수 생성 -> 전월 수정 주가 대입
4. 전년도의 수정주가 파생변수 생성 -> 전년도 수정 주가 대입
5. 구매 신호를 생성
6. 원본의 데이터에서 구호 신호에 따른 거래 내역 생성
7. 수익을 계산

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
df=pd.read_csv('../../csv/AMZN.csv', index_col='Date')
df.head()

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
1997-05-15,2.437500,2.500000,1.927083,1.958333,1.958333,72156000
1997-05-16,1.968750,1.979167,1.708333,1.729167,1.729167,14700000
1997-05-19,1.760417,1.770833,1.625000,1.708333,1.708333,6106800
1997-05-20,1.729167,1.750000,1.635417,1.635417,1.635417,5467200
1997-05-21,1.635417,1.645833,1.375000,1.427083,1.427083,18853200


In [3]:
#index 데이터 시계열로 변환
df.index=pd.to_datetime(df.index)

In [4]:
#index 데이터에서 년-월을 추출하여 STD-YM에 대입
df.index.strftime('%Y-%m')

Index(['1997-05', '1997-05', '1997-05', '1997-05', '1997-05', '1997-05',
       '1997-05', '1997-05', '1997-05', '1997-05',
       ...
       '2019-06', '2019-06', '2019-06', '2019-06', '2019-06', '2019-06',
       '2019-06', '2019-06', '2019-06', '2019-06'],
      dtype='object', name='Date', length=5563)

In [ ]:
ym_list=[]

for idx in df.index:
    ym=idx.strftime('%Y-%m')
    ym_list.append(ym)

ym_list

In [6]:
df['STD-YM']=ym_list

In [7]:
df.loc['1997-05-25' : '1991-06-05', ]


,Open,High,Low,Close,Adj Close,Volume,STD-YM
Date,,,,,,,


In [8]:
#현재 행의 STD-YM과 다음 행의 STD-YM이 다른 경우 : 월말
flag=df['STD-YM'] != df.shift(-1)['STD-YM']
df.loc[flag,]

,Open,High,Low,Close,Adj Close,Volume,STD-YM
Date,,,,,,,
1997-05-30,1.500000,1.510417,1.479167,1.500000,1.500000,2594400,1997-05
1997-06-30,1.510417,1.598958,1.479167,1.541667,1.541667,2746800,1997-06
1997-07-31,2.437500,2.437500,2.333333,2.395833,2.395833,1454400,1997-07
1997-08-29,2.364583,2.375000,2.322917,2.338542,2.338542,722400,1997-08
1997-09-30,4.000000,4.348958,3.802083,4.338542,4.338542,5254800,1997-09
...,...,...,...,...,...,...,...
2019-02-28,1635.250000,1651.770020,1633.829956,1639.829956,1639.829956,3025900,2019-02
2019-03-29,1786.579956,1792.859985,1776.630005,1780.750000,1780.750000,3320800,2019-03
2019-04-30,1930.099976,1935.709961,1906.949951,1926.520020,1926.520020,3506000,2019-04


In [9]:
#그룹화를 하고 마지막 데이터만 확인
month_last_df=df.groupby('STD-YM').tail(1)

In [10]:
#전월의 수정주가, 전년도의 수정주가 컬럼을 생성
month_last_df['BF-1M']=month_last_df.shift(1)['Adj Close'].fillna(0)
month_last_df['BF-12M']=month_last_df.shift(12)['Adj Close'].fillna(0)

month_last_df.iloc[10:15, ]

C:\Users\user\AppData\Local\Temp\ipykernel_19092\706146360.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  month_last_df['BF-1M']=month_last_df.shift(1)['Adj Close'].fillna(0)
C:\Users\user\AppData\Local\Temp\ipykernel_19092\706146360.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  month_last_df['BF-12M']=month_last_df.shift(12)['Adj Close'].fillna(0)


,Open,High,Low,Close,Adj Close,Volume,STD-YM,BF-1M,BF-12M
Date,,,,,,,,,
1998-03-31,7.114583,7.208333,6.979167,7.127600,7.127600,6565200,1998-03,6.416667,0.000000
1998-04-30,8.125000,8.166667,7.541667,7.645833,7.645833,22485600,1998-04,7.127600,0.000000
1998-05-29,7.156250,7.416667,7.125000,7.343750,7.343750,8641200,1998-05,7.645833,1.500000
1998-06-30,16.302084,16.916666,16.125000,16.625000,16.625000,21877200,1998-06,7.343750,1.541667
1998-07-31,19.083334,19.187500,18.187500,18.479166,18.479166,13440600,1998-07,16.625000,2.395833


In [13]:
#거래내역 추가 (_df, month_last_df)
#거래내역은 df에 추가
#month_last_df는 구매 신호(momentum_index)를 확인하기 위해 사용
for i in month_last_df.index:
    signal=''

    #절대 모멘텀의 계산식 : 전월 수정 주가 / 전년도 수정 주가 - 1
    momentum_index=month_last_df.loc[i,'BF-1M'] / month_last_df.loc[i,'BF-12M'] - 1
    #0보다 크고 무한대가 아닌 경우 : 구매 신호
    flag = (momentum_index > 0) & (momentum_index != np.inf)
    if flag:
        signal='buy'
    
    print(f'날짜 : {i}, 모멘텀 인덱스 : {momentum_index}, signal : {signal}')
    df.loc[i:,'trade']=signal

C:\Users\user\AppData\Local\Temp\ipykernel_19092\2645774783.py:8: RuntimeWarning: invalid value encountered in scalar divide
  momentum_index=month_last_df.loc[i,'BF-1M'] / month_last_df.loc[i,'BF-12M'] - 1
C:\Users\user\AppData\Local\Temp\ipykernel_19092\2645774783.py:8: RuntimeWarning: divide by zero encountered in scalar divide
  momentum_index=month_last_df.loc[i,'BF-1M'] / month_last_df.loc[i,'BF-12M'] - 1
C:\Users\user\AppData\Local\Temp\ipykernel_19092\2645774783.py:8: RuntimeWarning: divide by zero encountered in scalar divide
  momentum_index=month_last_df.loc[i,'BF-1M'] / month_last_df.loc[i,'BF-12M'] - 1
C:\Users\user\AppData\Local\Temp\ipykernel_19092\2645774783.py:8: RuntimeWarning: divide by zero encountered in scalar divide
  momentum_index=month_last_df.loc[i,'BF-1M'] / month_last_df.loc[i,'BF-12M'] - 1
C:\Users\user\AppData\Local\Temp\ipykernel_19092\2645774783.py:8: RuntimeWarning: divide by zero encountered in scalar divide
  momentum_index=month_last_df.loc[i,'BF-1M

날짜 : 1997-05-30 00:00:00, 모멘텀 인덱스 : nan, signal : 
날짜 : 1997-06-30 00:00:00, 모멘텀 인덱스 : inf, signal : 
날짜 : 1997-07-31 00:00:00, 모멘텀 인덱스 : inf, signal : 
날짜 : 1997-08-29 00:00:00, 모멘텀 인덱스 : inf, signal : 
날짜 : 1997-09-30 00:00:00, 모멘텀 인덱스 : inf, signal : 
날짜 : 1997-10-31 00:00:00, 모멘텀 인덱스 : inf, signal : 
날짜 : 1997-11-28 00:00:00, 모멘텀 인덱스 : inf, signal : 
날짜 : 1997-12-31 00:00:00, 모멘텀 인덱스 : inf, signal : 
날짜 : 1998-01-30 00:00:00, 모멘텀 인덱스 : inf, signal : 
날짜 : 1998-02-27 00:00:00, 모멘텀 인덱스 : inf, signal : 
날짜 : 1998-03-31 00:00:00, 모멘텀 인덱스 : inf, signal : 
날짜 : 1998-04-30 00:00:00, 모멘텀 인덱스 : inf, signal : 
날짜 : 1998-05-29 00:00:00, 모멘텀 인덱스 : 4.0972219999999995, signal : buy
날짜 : 1998-06-30 00:00:00, 모멘텀 인덱스 : 3.7635124835648686, signal : buy
날짜 : 1998-07-31 00:00:00, 모멘텀 인덱스 : 5.9391314002269775, signal : buy
날짜 : 1998-08-31 00:00:00, 모멘텀 인덱스 : 6.902003042921615, signal : buy
날짜 : 1998-09-30 00:00:00, 모멘텀 인덱스 : 2.217286590748689, signal : buy
날짜 : 1998-10-30 00:00:00, 모멘텀 인덱스 : 2.6598361

In [14]:
df['trade'].value_counts()

trade
buy    4033
       1520
Name: count, dtype: int64

In [16]:
#수익율, 누적 수익율 계산
df['rtn']=1
df.head()

,Open,High,Low,Close,Adj Close,Volume,STD-YM,trade,rtn
Date,,,,,,,,,
1997-05-15,2.437500,2.500000,1.927083,1.958333,1.958333,72156000,1997-05,NaN,1
1997-05-16,1.968750,1.979167,1.708333,1.729167,1.729167,14700000,1997-05,NaN,1
1997-05-19,1.760417,1.770833,1.625000,1.708333,1.708333,6106800,1997-05,NaN,1
1997-05-20,1.729167,1.750000,1.635417,1.635417,1.635417,5467200,1997-05,NaN,1
1997-05-21,1.635417,1.645833,1.375000,1.427083,1.427083,18853200,1997-05,NaN,1


In [ ]:
for i in df.index:
    #매수 가격을 형성
    if (df.shift().loc[i,'trade']=='')  & (df.loc[i,'trade']=='buy'):
        buy = df.loc[i, 'Adj Close']
        print(f'매수일 : {i}, 매수가 : {buy}')
    # 매도 가격을 형성
    elif (df.shift().loc[i,'trade']=='buy')  & (df.loc[i,'trade']==''):
        sell = df.loc[i, 'Adj Close']
        #수익율 게산
        rtn = sell / buy
        df.loc[i,'rtn']=rtn
        print(f'매도일 : {i}, 매도가 : {sell}, 수익율 : {rtn}')

In [21]:
df['acc_rtn']=df['rtn'].cumprod()
df.tail()

,Open,High,Low,Close,Adj Close,Volume,STD-YM,trade,rtn,acc_rtn
Date,,,,,,,,,,
2019-06-18,1901.349976,1921.670044,1899.790039,1901.369995,1901.369995,3895700,2019-06,buy,1.0,86.522946
2019-06-19,1907.839966,1919.579956,1892.469971,1908.790039,1908.790039,2895300,2019-06,buy,1.0,86.522946
2019-06-20,1933.329956,1935.199951,1905.800049,1918.189941,1918.189941,3217200,2019-06,buy,1.0,86.522946
2019-06-21,1916.099976,1925.949951,1907.579956,1911.300049,1911.300049,3920300,2019-06,buy,1.0,86.522946
2019-06-24,1912.660034,1916.859985,1901.329956,1907.953857,1907.953857,1243601,2019-06,buy,1.0,86.522946


In [22]:
df.iloc[-1,-1]

np.float64(86.52294619753461)

In [23]:
#buyandhold 수익률
buy = df['Adj Close'][0]
sell=df['Adj Close'][-1]
print(sell/buy)

974.2744757914


C:\Users\user\AppData\Local\Temp\ipykernel_19092\2757622830.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  buy = df['Adj Close'][0]
C:\Users\user\AppData\Local\Temp\ipykernel_19092\2757622830.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sell=df['Adj Close'][-1]


### 절대 모멘텀 함수화
1. STD-YM 생성하는 함수
    - 매개변수
        - 데이터
        - 기준이 되는 컬럼
    - 데이터프레임을 깊은 복사
    - 컬럼에 Date가 포함되어 있는가를 확인
        - 포함되어 있다면 Date를 인덱스로 변경
    - 인덱스를 시계열 데이터로 변경
    - tz 제거
    - 데이터에서 결측치와 무한를대 제거
    - 기준이 되는 컬럼을 제외하고 모두 제거
    - STD-YM 컬럼을 생성하여 인덱스에서 년도-월 데이터를 추출한다.
    - 수정된 데이터 프레임을 되돌려준다.

In [24]:
def create_ym(_df, _col='Adj Close'):
    df=_df.copy()
    #Date가 컬럼에 포함되어 있는가?
    if 'Date' in df.columns:
        df.set_index('Date', inplace=True)
    df.index=pd.to_datetime(df.index)
    df.index=df.index.tz_localize(None)
    #결측치, 무한대 데이터 제거
    flag=df.isin([np.nan, np.inf, -np.inf]).any(axis=1)
    df=df.loc[~flag, [_col]]
    #파생 변수 STD-YM
    df['STD-YM']=df.index.strftime('%Y-%m')

    return df

In [25]:
df2=pd.read_csv('../../csv/AAPL.csv')

ym_df=create_ym(df2)
ym_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 9713 entries, 1980-12-12 to 2019-06-24
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Close  9713 non-null   float64
 1   STD-YM     9713 non-null   object 
dtypes: float64(1), object(1)
memory usage: 227.6+ KB


In [28]:
np.nan == np.nan

False

In [29]:
np.nan in [np.nan]

True

2. 월말 데이터를 생성하여 BF1, BF2 컬럼을 생성하는 함수
    - 매개변수
        - create_ym의 결과를 받아주는 데이터
        - 시작 시간 : 2010-01-01
        - 종료 시간 : 현재 시간
        - 모멘텀 기간 : 12
        - 기준 시점 : 1 (1인 경우 월말, 0인 경우 월초)
    - 기준 시점의 값에 따라서 (월말 | 월초) 데이터만 모은 새로운 데이터 프레임을 생성
    - 생성된 데이터프레임에서 BF1 컬럼을 생성하여 전월을 데이터를 대입
    - BF2 컬럼을 생성하여 모멘텀 기간(6 : 6개월 전) 전의 데이터를 대입
    - 결측치는 0으로 대입
    - 데이터 프레임을 시작시간과 종료시간으로 데이터 필터링
    - 결과를 리턴 (월말 | 월초 데이터프레임)

In [34]:
def create_month(
    _df,
    _start='2010-01-01',
    _end=datetime.now(),
    _momentum=12,
    _last=1
):
    #_last 값에 따라서 월말, 월초
    if _last == 1:
        df = _df.groupby('STD-YM').tail(1)
    elif _last == 0:
        df = _df.groupby('STD-YM').head(1)
    else:
        return '_last의 값은 0과 1만 가능합니다.'
    #기준이 되는 컬럼의 이름을 변수로 저장
    col=_df.columns[0]
    #전월의 데이터를 BF1에 대입
    df['BF1']=df.shift(1)[col].fillna(0)
    df['BF2']=df.shift(_momentum)[col].fillna(0)

    #시작시간과 종료시간을 기준으로 데이터를 필터링
    df=df.loc[_start : _end,]
    return df


In [ ]:
month_df=create_month(ym_df, _momentum=6)
month_df.head()

3. 거래내역을 추가하고 수익을 계산하는 함수
    - 매개변수
        - ym_df가 대입이 될 수 있는 변수
        - month_df가 대입이 됳 수 있는 변수
        - 모멘텀 스코어 : 1
    - _df1을 깊은 복사 (df)
    - df에 'trade' 컬럼을 생성하여 빈 텍스트 대입
    - df에 'rtn' 컬럼을 생성하여 1을 대입
    - _df2를 이용하여 모멘텀 인덱스를 생성하고 0보다 크고 무한대가 아닌 경우에는 df에 보유 내역을 추가
    - 수익율 계산
    - 누적 수익율 계산
    - 데이터프레임과 최종 누적 수익율을 되돌려준다.

In [41]:
def create_rtn(_df1,_df2, _score=1):
    df=_df1.copy()

    df['trade']=''
    df['rtn']=1
    col=df.columns[0]

    #_df2를 이용해서 거래내역을 생성
    for i in _df2.index:
        signal=''
        #모멘텀 계산
        momentum_index=_df2.loc[i,'BF1']/_df2.loc[i,'BF2']-_score
        flag=(momentum_index > 0) & (momentum_index != np.inf)
        if flag:
            signal ='buy'

        #거래 내역 생성
        df.loc[i:, 'trade']=signal
        print(f'날짜 : {i}, 모멘텀 인덱스 : {momentum_index}, signal : {signal}')
    
    #수익율 계산
    for i in df.index:
        if (df.shift().loc[i,'trade'] == '') & (df.loc[i,'trade'] == 'buy'):
            buy=df.loc[i,col]
            print(f'매수일 : {i}, 매수가 : {buy}')
        elif (df.shift().loc[i,'trade'] == 'buy') & (df.loc[i,'trade'] == ''):
            sell = df.loc[i,col]
            rtn=sell/buy
            df.loc[i,'rtn']=rtn
            print(f'매도일 : {i}, 매도가 : {buy}, 수익율 : {rtn}')

    #누적 수익율 계산
    df['acc_rtn']=df['rtn'].cumprod()

    acc_rtn=df.iloc[-1,-1]

    return df, acc_rtn

In [ ]:
rtn_df, acc_rtn = create_rtn(ym_df,month_df, _score=1.1)

print(acc_rtn)